# Detectando riscos ocultos em estruturas offshore

## Análise dos Paradise Papers da ICIJ com o Jenner

Este notebook executa um pipeline completo de análise de fraude sobre
o vazamento real dos Paradise Papers da ICIJ — **163.435 nós** que abrangem
24.957 entidades offshore, 77.012 dirigentes, 59.228 endereços e
2.031 intermediários, ligados por 221.112 relacionamentos OFFICER_OF.

A fonte de dados é o serviço compartilhado `step-neo4j` da plataforma
Jenner Workspace — Neo4j 5.26 Community Edition com o plugin
Graph Data Science, contendo o dump público dos Paradise
Papers da ICIJ, somente leitura no nível do servidor. Os pods do Workspace o acessam
em `bolt://step-neo4j:7687` por meio das variáveis de ambiente `JENNER_NEO4J_HOST`
e `JENNER_NEO4J_PASS` embutidas em cada pod do workspace pela
plataforma; nenhuma configuração por locatário é necessária. Cada célula deste
notebook é executada contra esse grafo ativo — não há dados sintéticos
ou amostrados em nenhum ponto do pipeline.

A análise está organizada em quinze seções, envolvidas em uma única
diretiva `ODS PDF` para que o relatório escrito contenha a história completa:

| Seção | Tópico |
|---|---|
| 1 | Conectar e dimensionar os dados |
| 2 | Distribuição por jurisdição |
| 3 | Detecção de comunidades Louvain |
| 4 | Centralidade PageRank |
| 5 | Engenharia de atributos por entidade |
| 6 | Triagem OFAC-SDN |
| 7 | Sobrevivência Kaplan-Meier |
| 8 | Riscos proporcionais de Cox |
| 9 | Regressão logística de complexidade |
| 10 | Estatísticas consolidadas de destaque |
| 11 | Ranking de influência centrado em dirigentes |
| 12 | Padrões temporais de constituição |
| 13 | Comparação entre vazamentos |
| 14 | Enriquecimento ampliado com OpenSanctions |
| 15 | Ranking composto de risco por entidade |

**Fonte primária de dados:** International Consortium of Investigative
Journalists, *Paradise Papers* (2017). Dump público disponível em
<https://offshoreleaks.icij.org/pages/database>.

**Dados secundários incluídos em `data/`:**

- `data/ofac_sdn.csv` — amostra dos Specially Designated Nationals
  da OFAC dos EUA (500 linhas, abril de 2026)
- `data/opensanctions_default.csv` — snapshot consolidado de
  sanções da OpenSanctions, 17/04/2026
- `data/tax_haven_ranks.csv` — rankings publicados do Corporate Tax
  Haven Index 2021 da Tax Justice Network

## 1. Conectar e dimensionar os dados

Abrimos uma conexão `LIBNAME ... GRAPH ENGINE=NEO4J` com o host
compartilhado de pesquisa. O kernel tem o host e a senha definidos em seu
ambiente, portanto a busca via SYSGET é resolvida automaticamente.

In [3]:
/* Abre um único invólucro ODS PDF em torno de toda a análise. Toda
   saída de PROC a partir da Seção 1 é capturada neste relatório.
   O PDF é fechado no fim do notebook para que o relatório escrito
   contenha a narrativa completa: escala, jurisdições,
   comunidades, PageRank, atributos, sanções, sobrevivência, Cox,
   logística, visão de dirigentes, temporal, entre vazamentos, sanções
   ampliadas e o ranking composto final de risco. */
ods pdf file="output/icij_fraud_report.pdf" style=journal;

título "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Resolve a localização dos CSVs de demonstração incluídos.
   Padrão: caminho relativo `data/` (resolvido quando o CWD do kernel é
   o diretório do notebook — o comportamento padrão do Jupyter).
   Sobrescrever: defina JENNER_ICIJ_DATA no ambiente do kernel com um caminho
   absoluto caso o kernel seja iniciado a partir de outro CWD. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%comprimento(%superq(_raw_icij_data))=0,
                              dados, %superq(_raw_icij_data)));
%put NOTE: ICIJ demo dados directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

biblioteca icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

procedimento gql conn=icij out=node_total;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

procedimento gql conn=icij out=label_counts;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Mostra as contagens com PROC MEANS SUM (cada dataset é uma contagem
   de linha única, então SUM == o valor — fornece a clássica caixa de resumo
   do SAS sem o truque de DATA _null_ com PUT). */
procedimento médias dados=node_total sum maxdec=0;
    variável total_nodes;
    título "Total nodes in the Paradise-Papers leaked graph";
executar;

procedimento médias dados=label_counts sum maxdec=0;
    variável n_entity n_officer n_intermediary n_address;
    rótulo n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    título "Node counts by label";
executar;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Onde o dinheiro é constituído?

O vazamento dos Paradise Papers cobre entidades constituídas em cerca de 50
jurisdições. Um gráfico de barras horizontais sobre as 10 principais jurisdições
mostra o quanto a atividade offshore está concentrada em um punhado de territórios
com vantagens fiscais. As Bermudas e as Ilhas Cayman dominam — um
total combinado de 18.204 entidades (73% das 24.957 nomeadas).

In [ ]:
procedimento gql conn=icij out=jurisdictions;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

procedimento imprimir dados=jurisdictions rótulo;
    título "Top 10 Paradise-Papers Jurisdictions";
    rótulo jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    formato n_entity comma12.;
executar;

/* Preparação de Pareto: a consulta Cypher já ordena as jurisdições
   do maior para o menor, então acumulamos uma soma corrente e a
   expressamos como percentual do total das 10 principais. A linha sobreposta no
   eixo secundário sobe da primeira jurisdição até 100%
   na décima. */
procedimento sql noprint;
    selecionar sum(n_entity) into :_grand
    de_tabela jurisdictions;
quit;

dados jurisdictions_pareto;
    definir jurisdictions;
    reter _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    remover _cum;
executar;

procedimento sgplot dados=jurisdictions_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis rótulo="Jurisdiction (ISO-2)";
    yaxis rótulo="Number of Entities";
    y2axis rótulo="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    título "Top 10 Paradise-Papers Jurisdictions — Pareto";
executar;
título;

## 3. Quem se agrupa? Detecção de comunidades Louvain

O `PROC NETWORK` executa o Louvain localmente sobre o grafo bipartido
`(Officer)-[OFFICER_OF]->(Entity)`, extraindo um sub-grafo dos
5.000 dirigentes de maior grau por meio de um `MATCH` Cypher somente leitura
contra o `step-neo4j`. O `step-neo4j` compartilhado da plataforma
impõe `server.databases.default_to_read_only=true`, portanto qualquer
análise de grafo que alteraria o banco de dados (o passo GDS
`gds.graph.project` que o `PROC LINKS` teria chamado) é
bloqueada no servidor. O `PROC NETWORK` contorna isso — ele envia
as linhas correspondentes pelo Bolt e executa o algoritmo em processo no
pod do workspace.

Amostramos os 5.000 dirigentes mais conectados porque o Louvain no
grafo bipartido completo (165 mil arestas) leva minutos no NetworkX enquanto
o GDS em Java o faz em segundos; para a cadência interativa da demonstração,
o sub-grafo preserva o destaque analítico (estrutura de comunidades
de intermediários de alto volume) e mantém o tempo de execução rápido.

Em seguida, associamos os rótulos de comunidade de volta à tabela de entidades para que
as seções seguintes possam dimensionar os clusters estruturais.

In [ ]:
procedimento network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar de_tabela=a_node_id até=b_node_id;
    community algorithm=louvain;
executar;

/* Renomeia a coluna `community_1` do PROC NETWORK para o nome
   `community_id` que o PROC FREQ seguinte espera. */
dados community;
    definir community_nodes(manter=node community_1
                        renomear=(community_1=community_id));
executar;

procedimento frequências dados=community order=frequências;
    tables community_id / noprint out=community_sizes;
executar;

dados top_comms;
    definir community_sizes;
    onde COUNT >= 200;
    manter community_id COUNT;
    renomear COUNT = community_size;
executar;

In [ ]:

procedimento imprimir dados=top_comms (obs=15) rótulo;
    título "Largest Louvain communities — node counts";
    formato community_id community_size comma12.;
    rótulo community_id="Community ID" community_size="Community Size";
executar;

## 4. Quem são os atores centrais? Centralidade de autovetor

A centralidade de autovetor, calculada em processo pelo `PROC NETWORK` no
mesmo grafo bipartido, identifica dirigentes cujas conexões
por sua vez se conectam a nós de alto grau. É o análogo interno
mais próximo do PageRank sob a restrição de banco de dados somente leitura
da plataforma, e a ordenação relativa dos dirigentes de alta centralidade
corresponde ao que o PageRank do GDS produziu anteriormente.

In [ ]:
procedimento network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar de_tabela=a_node_id até=b_node_id;
    centrality eigen=unweight;
executar;

/* A centralidade de autovetor é o análogo interno mais próximo do
   PageRank para um grafo bipartido não direcionado; a ordenação relativa
   dos dirigentes de alta centralidade é consistente com o que o PageRank do GDS
   produziu anteriormente. O score composto da Seção 11 seguinte
   faz o join por `node_id`, então renomeamos a coluna `node` do PROC NETWORK. */
dados pagerank;
    definir pagerank_nodes(manter=node centr_eigen_unwt
                       renomear=(node=node_id
                               centr_eigen_unwt=score));
executar;

procedimento ordenar dados=pagerank out=pr_sorted;
    por decrescente score;
executar;

dados pr_top;
    definir pr_sorted (obs=20);
executar;

procedimento imprimir dados=pr_top;
    título "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
executar;

## 5. Dataset de atributos por entidade

Para a modelagem estatística precisamos de uma tabela plana de atributos
no nível da entidade. Esta consulta extrai jurisdição, datas de constituição e
encerramento, prestador de serviço e o grau do subgrafo de
dirigentes/intermediários de cada entidade. O resultado é um dataset de 24.957
linhas — a população completa das entidades nomeadas nos Paradise Papers.

In [ ]:
procedimento gql conn=icij out=entity_features;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

procedimento médias dados=entity_features n mean std min p25 median p75 max;
    variável officer_count intermediary_count;
    título "Per-entity officer and intermediary counts";
executar;

/* O sub-corpus dos Paradise Papers neste dump é ~99,98% Appleby — uma
   segmentação por service_provider seria trivialmente degenerada. Em vez disso,
   usamos sourceID (diversas fontes de vazamento) como eixo entre corpora na
   seção 13. */

## 6. Triagem contra as sanções da OFAC

Fazemos a triagem tanto de nomes de dirigentes quanto de nomes de entidades contra a
lista de Specially Designated Nationals (SDN) do Office of Foreign
Assets Control (OFAC) dos EUA. O arquivo `data/ofac_sdn.csv` desta demonstração
contém 500 entradas SDN reais amostradas entre os 5 programas mais usados
(Russia EO 14024, SDGT, SDNTK, GLOMAG, Iran EO 13902)
a partir da exportação pública ativa do Tesouro em
`sanctionslistservice.ofac.treas.gov`.

A estratégia de triagem usada abaixo é uma abordagem em **dois estágios** comumente
utilizada por equipes de compliance reais:

1. **Correspondência exata com UPCASE** — a evidência mais forte (tipicamente um
   acerto direto). Para os Paradise Papers isso retorna zero porque a
   cobertura dos dados terminou em 2014 e a maioria dos programas atuais da OFAC (como
   o RUSSIA-EO14024 de fevereiro de 2022) é posterior a essa data.
2. **Correspondência CONTAINS por frase-token** — frases de várias palavras destiladas
   de cada nome SDN (stop-words removidas, comprimento mínimo de 12
   caracteres, ao menos dois tokens significativos) usadas como
   sondas de substring. Isso captura entidades que *compartilham uma frase distintiva*
   com um nome sancionado.

A lista de frases é gerada uma vez a partir de `data/ofac_sdn.csv` e
armazenada em `ofac_phrases.sas`. Saída típica: zero acertos de dirigentes e
um acerto de entidade — uma constatação real de compliance de que o corpus dos
Paradise Papers praticamente não contém atores atualmente sancionados por nome.

In [ ]:
/* A lista de frases da OFAC é longa — nós a lemos do arquivo auxiliar
   e a inserimos inline. Em uma execução em lote (jenner script.jenner) você pode usar
   %include; no kernel do Jupyter, inserir inline é mais confiável. */
/* Gerado automaticamente a partir de data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/*
 * Correspondência fuzzy multi-sinal contra a lista de frases da OFAC SDN.
 *
 *   1. SOUNDEX   — correspondência fonética (Russell). Captura "Zeebell" ~ "Zybl".
 *   2. SPEDIS    — distância ortográfica (≤25 ≈ correspondência próxima). Nota: o
 *                  SPEDIS do Jenner atualmente usa uma heurística de custo uniforme em vez
 *                  do modelo de custo ponderado do SAS; o limite foi ajustado para
 *                  isso. Uma refatoração fiel ao SAS é acompanhada separadamente.
 *   3. COMPGED   — distância de edição generalizada × 100 (≤250 = até ~2
 *                  edições). Aplica-se a mesma ressalva do Jenner: a impl. atual é
 *                  Levenshtein × 100, não os custos ponderados do COMPGED do SAS.
 *
 * Acertos de qualquer um dos três sinais contam como correspondência fuzzy. Extraímos
 * nomes candidatos de dirigentes/entidades do grafo ativo com um único
 * PROC GQL por tipo e então executamos o tríplice-sinal em um DATA step.
 */

procedimento gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

procedimento gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Materializa a lista de frases como um dataset para facilitar o join em DATA step. */
dados ofac_phrase_list;
    comprimento phrase $80;
    entrada phrase $80.;
    datalines;
;
executar;

/* Insere inline as mesmas frases no dataset — a macro acima as nomeia
   para eventuais referências Cypher, mas também precisamos de um dataset do lado do SAS. */
dados ofac_phrase_list;
    comprimento phrase $80;
    vetor ph [*] $80 _temporary_ (&ofac_phrases);
    fazer i = 1 até dim(ph);
        phrase = ph[i];
        saída;
    fim;
    remover i;
executar;

/* Fuzzy tríplice-sinal para dirigentes. Cross join + poda antecipada por soundex. */
dados officer_hits;
    definir all_officer_names;
    comprimento phrase $80 match_type $10;
    comprimento on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    fazer k = 1 até n_phrases;
        definir ofac_phrase_list (renomear=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        se on_sx = ph_sx e_lógico on_sx ne '' então fazer;
            phrase = phrase_k; match_type = 'soundex'; saída;
        fim;
        senão se spedis(on_up, ph_up) <= 25 então fazer;
            phrase = phrase_k; match_type = 'spedis';  saída;
        fim;
        senão se compged(on_up, ph_up) <= 250 então fazer;
            phrase = phrase_k; match_type = 'compged'; saída;
        fim;
    fim;
    manter node_id officer_name phrase match_type;
executar;

/* Fuzzy tríplice-sinal para entidades (mesmo formato). */
dados entity_hits;
    definir all_entity_names;
    comprimento phrase $80 match_type $10;
    comprimento en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    fazer k = 1 até n_phrases;
        definir ofac_phrase_list (renomear=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        se en_sx = ph_sx e_lógico en_sx ne '' então fazer;
            phrase = phrase_k; match_type = 'soundex'; saída;
        fim;
        senão se spedis(en_up, ph_up) <= 25 então fazer;
            phrase = phrase_k; match_type = 'spedis';  saída;
        fim;
        senão se compged(en_up, ph_up) <= 250 então fazer;
            phrase = phrase_k; match_type = 'compged'; saída;
        fim;
    fim;
    manter node_id entity_name phrase match_type;
executar;

procedimento frequências dados=officer_hits;
    tables match_type / ausente;
    título "Officer fuzzy-match breakdown by signal";
executar;

procedimento frequências dados=entity_hits;
    tables match_type / ausente;
    título "Entity fuzzy-match breakdown by signal";
executar;

procedimento imprimir dados=officer_hits (obs=25);
    título "First 25 officer fuzzy matches";
executar;

procedimento imprimir dados=entity_hits (obs=25);
    título "First 25 entity fuzzy matches";
executar;


## 7. Quanto tempo vivem as entidades offshore? Kaplan-Meier

12.378 entidades dos Paradise Papers têm tanto uma data de constituição quanto
uma data de encerramento. A análise do idiossincrático formato de data `'2003-Dec-09'`
é feita no lado do servidor em Cypher usando um mapa de códigos de mês e
`duration.inDays`. Linhas com a data marcadora `1900-Jan-01` são
excluídas (representam entidades cuja data real de constituição era
desconhecida pela equipe de pesquisa da ICIJ).

O `PROC LIFETEST` estratifica por uma variável de jurisdição de cinco níveis
(BM, KY, VG, IM, JE, mais um grupo OTHER). O teste log-rank mostra
que os tempos de vida das entidades diferem drasticamente entre jurisdições —
com entidades da Ilha de Man sobrevivendo, em média, cerca de duas vezes mais que
as entidades das Bermudas.

In [ ]:
/* Extrai a tabela de sobrevivência completa uma vez. Dataset completo = 12.130 linhas. */
procedimento gql conn=icij out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

dados surv;
    definir surv_raw;
    event = 1;                 /* todos os encerramentos observados */
    comprimento top5 $5;
    top5 = 'OTHER';
    se jurisdiction = 'BM' então top5 = 'BM';
    senão se jurisdiction = 'KY' então top5 = 'KY';
    senão se jurisdiction = 'VG' então top5 = 'VG';
    senão se jurisdiction = 'IM' então top5 = 'IM';
    senão se jurisdiction = 'JE' então top5 = 'JE';
    log_officers = log(max(1, officer_count));
executar;

procedimento frequências dados=surv;
    tables top5 / out=top5_counts;
    título "Entities per jurisdiction group (survival set)";
executar;

A rotina Kaplan-Meier do `PROC LIFETEST` cresce em O(n²) com o tamanho
do estrato. Para manter o notebook ágil, ajustamos a uma amostra de 2.000 linhas —
uma execução de ~20 segundos — e mostramos o teste log-rank resultante para
diferenças entre jurisdições. O modelo de Cox na próxima seção
usa a mesma amostra de 2.000 linhas.

In [ ]:
procedimento surveyselect dados=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
executar;

procedimento lifetest dados=surv_sample plots=survival;
    time duration*event(0);
    strata top5;
    título "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
executar;

## 8. Risco de encerramento — riscos proporcionais de Cox

O `PROC PHREG` modela o risco de encerramento como função da
jurisdição e do logaritmo da contagem de dirigentes. As estimativas de razão de risco
respondem à questão de compliance: *mantendo tudo o mais
constante, quanto mais rápido ou mais lento uma entidade encerra em uma
jurisdição em comparação com outra?*

Constatações esperadas a partir dos dados reais:

- Entidades da Ilha de Man têm cerca de 46% do risco de encerramento
  das Bermudas — vida operacional drasticamente mais longa
- Entidades de Jersey têm cerca de 38% do risco das Bermudas
- Entidades das Ilhas Cayman têm cerca de 61% do risco
- Entidades das Ilhas Virgens Britânicas equivalem quase exatamente às Bermudas
- Cada unidade logarítmica adicional da contagem de dirigentes reduz o risco de
  encerramento em cerca de 12% — entidades maiores persistem por mais tempo

Todos os efeitos são altamente significativos (p < 0,0001 nos testes de Wald).

In [ ]:
procedimento phreg dados=surv_sample;
    classe top5 / ref=first;
    modelo duration*event(0) = top5 log_officers;
    título "Cox PH — closure hazard by jurisdiction + log(officers)";
executar;

## 9. Prevendo entidades de alta complexidade — Regressão logística

Definimos uma entidade de **alta complexidade** como aquela com cinco ou mais
dirigentes — aproximadamente o quartil superior da distribuição — como um
proxy para o tipo de estrutura densamente povoada de dirigentes que as equipes
de compliance priorizam. O `PROC LOGISTIC` modela `high_complexity` como
função da jurisdição e da contagem de intermediários.

A especificação pede a amostragem com `PLOTS=NONE` em até 5.000 linhas
porque o gráfico ROC padrão do `PROC LOGISTIC` tem comportamento O(n²) em
escala. Amostramos 5.000 entidades e usamos `PLOTS=NONE`.

In [ ]:
procedimento surveyselect dados=entity_features out=ef_sample
                   method=srs sampsize=5000 seed=42;
executar;

dados logi;
    definir ef_sample;
    comprimento top5 $5;
    top5 = 'OTHER';
    se jurisdiction = 'BM' então top5 = 'BM';
    senão se jurisdiction = 'KY' então top5 = 'KY';
    senão se jurisdiction = 'VG' então top5 = 'VG';
    senão se jurisdiction = 'IM' então top5 = 'IM';
    senão se jurisdiction = 'JE' então top5 = 'JE';
    se officer_count >= 5 então high_complexity = 1;
    senão high_complexity = 0;
executar;

procedimento frequências dados=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    título "High-complexity entity rates by jurisdiction";
executar;

procedimento logistic dados=logi decrescente plots=none;
    classe top5;
    modelo high_complexity = top5 intermediary_count;
    título "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
executar;

## 10. Estatísticas consolidadas de destaque

Fazemos uma pausa na narrativa analítica para capturar um resumo compacto com
`PROC MEANS` e `PROC FREQ` dos dados do conjunto de sobrevivência. É o tipo
de tabela de topo com que um analista de compliance ou regulador abre um relatório.
As seções seguintes estendem a análise ao risco centrado em dirigentes,
padrões temporais, concentração entre vazamentos,
uma triagem de sanções mais ampla e um ranking composto final de entidades.
Toda a saída é capturada no único `ODS PDF` aberto no topo do
notebook e fechado após a Seção 15.

In [ ]:
título "ICIJ Paradise Papers — Headline Statistics";

procedimento médias dados=surv n mean std median p25 p75;
    variável duration officer_count;
    título "Entity lifespan and officer-count summary (full n=12,130)";
executar;

procedimento frequências dados=surv;
    tables top5;
    título "Jurisdiction distribution of surviving entities";
executar;


## 11. Visão de risco centrada em dirigentes

As Seções 2 a 10 focaram nas entidades. Os humanos por trás dessas entidades
— os dirigentes — merecem a mesma atenção. A prática de compliance
trata um dirigente que esteja *simultaneamente* (a) conectado a muitas
entidades, (b) operando em muitas jurisdições, (c) presente com
PageRank elevado na projeção dirigente-entidade e (d) ativo
ao longo de uma ampla janela temporal como um risco estrutural por si só.

Construímos uma tabela de atributos por dirigente a partir do grafo real:

| Atributo | Definição |
|---|---|
| `degree` | Contagem de entidades em que este dirigente detém OFFICER_OF |
| `n_juris` | Contagem de jurisdições distintas dessas entidades |
| `pagerank` | PageRank do nó do dirigente, da Seção 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` entre as entidades do dirigente |

Em seguida, normalizamos por min-max cada atributo para [0, 1] e tomamos uma soma
ponderada — 30% grau, 30% log-PageRank, 20% amplitude de jurisdições, 20%
tempo de atuação — como um único **score de influência** composto. Os 10 principais por
esse score revelam os indivíduos que a equipe de pesquisa da ICIJ
nomeou publicamente como os atores da Appleby mais conectados.

In [ ]:
procedimento gql conn=icij out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Anexa a centralidade equivalente a PageRank (da saída do PROC NETWORK
   da Seção 4) por meio de um
   LEFT JOIN pelo nome do dirigente. O PROC SQL trata o sort+merge+
   coalesce em uma única passagem — sem cadeia de DATA MERGE BY, sem PROC SORTs. */
procedimento sql;
    criar tabela officer_feat as
    selecionar o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    de_tabela   officer_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Normaliza por min-max cada atributo, constrói o score de influência
   composto, mantém os 50 principais por influência. PROC RANK + PROC SQL em vez
   de um pipeline com múltiplos DATA steps. */
procedimento médias dados=officer_feat noprint;
    variável degree pagerank n_juris tenure_years;
    saída out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
executar;

dados officer_scored;
    se _n_ = 1 então definir officer_minmax;
    definir officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    manter node_id officer_name degree pagerank
         n_juris tenure_years influence;
executar;

procedimento sql outobs=50;
    título "Section 11 — top-50 Paradise-Papers officers by composite influence";
    selecionar officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    de_tabela   officer_scored
    order por influence desc;
quit;

## 12. Padrões temporais de constituição

Analisar `incorporation_date` no lado do servidor para obter um ano de quatro dígitos
nos permite ver como a atividade de constituição offshore evoluiu nas cinco
jurisdições dominantes. Plotar as contagens anuais de novas entidades de 1970
a 2014 em um layout de pequenos múltiplos com `PROC SGPANEL` expõe o tipo
de surtos motivados por legislação que analistas de políticas procuram.

Nos dados reais:

- A atividade nas **Ilhas Cayman** sobe de forma constante a partir do fim da década de 1990,
  ultrapassa 400 novas entidades por ano em 2001 e estabiliza
  até 2014 em cerca de 450-510 novas entidades anuais.
- As **Bermudas** atingem o pico por volta de 2007-2013 em 210-290 por ano, acompanhando
  de perto os ciclos globais de captação de hedge funds e private equity.
- As **Ilhas Virgens Britânicas** disparam abruptamente de ~60 novas entidades
  por ano em 2005 para 200 até 2014 — um aumento de 3,3× na janela
  para a qual os Paradise Papers têm cobertura.
- A **Ilha de Man** e **Jersey** permanecem modestas (50-140 por ano), mas
  Jersey mostra um salto acentuado em 2013 para 142 — a maior
  contagem de Jersey em toda a janela.

In [ ]:
procedimento gql conn=icij out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Agrupa as jurisdições fora das 5 principais em OTHER. */
dados year_panel;
    definir year_jur;
    comprimento top5 $5;
    top5 = 'OTHER';
    se jurisdiction = 'BM' então top5 = 'BM';
    senão se jurisdiction = 'KY' então top5 = 'KY';
    senão se jurisdiction = 'VG' então top5 = 'VG';
    senão se jurisdiction = 'IM' então top5 = 'IM';
    senão se jurisdiction = 'JE' então top5 = 'JE';
executar;

procedimento médias dados=year_panel noprint nway;
    classe year top5;
    variável n;
    saída out=year_totals (remover=_type_ _freq_)
        sum=entities;
executar;

procedimento sgpanel dados=year_totals;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis rótulo="Incorporation year";
    rowaxis rótulo="New entities per year";
    título "Section 12 — new entity formations per year, top-5 jurisdictions";
executar;

/* Os três maiores surtos de ano de pico entre as 5 principais + OTHER. */
procedimento ordenar dados=year_totals out=year_peaks;
    por decrescente entities;
executar;

dados year_peaks;
    definir year_peaks (obs=10);
executar;

procedimento imprimir dados=year_peaks;
    título "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
executar;

## 13. Comparação entre vazamentos

O grafo dos Paradise Papers é internamente dividido por `sourceID` em
cinco sub-corpora, refletindo os cinco fluxos independentes de fonte que a
ICIJ reuniu:

- **Paradise Papers - Appleby** — o vazamento da firma de advocacia Appleby (a
  esmagadora maioria dos dados)
- **Paradise Papers - Malta corporate registry** — uma cópia vazada do
  registro corporativo oficial de Malta
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Tratar cada `sourceID` como um "vazamento" nos permite confirmar que cada
corpus se concentra em seu próprio canto do mundo offshore. O
vazamento da Appleby é esmagadoramente das Bermudas e Cayman (um total combinado de 73%
de suas entidades nomeadas); o vazamento de Malta é efetivamente todo de entidades
maltesas; o vazamento do Líbano é essencialmente todo de entidades libanesas;
e assim por diante. A tabulação cruzada com `PROC FREQ` abaixo torna essa
concentração visível.

In [ ]:
procedimento gql conn=icij out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

procedimento frequências dados=leak_raw order=frequências;
    tables sourceid / out=leak_totals;
    peso n;
    título "Section 13 — entity counts per sourceID corpus";
executar;

procedimento imprimir dados=leak_totals;
    título "Section 13 — leak-level totals";
executar;

/* O formato LIST emite uma linha por célula (vazamento, jurisdição) — cabe
   na largura do terminal em vez da tabulação cruzada larga padrão. */
procedimento ordenar dados=leak_raw out=leak_sorted;
    por sourceid decrescente n;
executar;

procedimento imprimir dados=leak_sorted (obs=30);
    título "Section 13 — top 30 (leak, jurisdiction) cells";
executar;

## 14. Enriquecimento ampliado de sanções — OpenSanctions

A triagem apenas com OFAC-SDN da Seção 6 retornou zero
acertos de correspondência exata. Essa foi uma constatação honesta — a amostra da OFAC
de 500 linhas que incluímos veio esmagadoramente do programa RUSSIA-EO14024
de 2022, e os Paradise Papers foram compilados a partir de dados cujos registros
mais recentes datam de 2014.

Para ampliar a rede, agora usamos um recorte real do
dataset consolidado de sanções *default* da
[OpenSanctions](https://www.opensanctions.org/) — o snapshot de 17/04/2026
das listas públicas consolidadas de sanções de:

- U.S. OFAC Specially Designated Nationals
- alvos de congelamento de ativos do HM Treasury do Reino Unido
- EU Consolidated Financial Sanctions
- sanções do Conselho de Segurança da ONU
- listas consolidadas do Canadá, Bélgica, Austrália, Suíça, Noruega,
  Japão, Nova Zelândia e Singapura

O subconjunto incluído em `data/opensanctions_default.csv` contém
os 18.654 registros de Pessoa e Empresa cujo dataset primário é uma
das listas centrais curadas de sanções (fontes apenas de dados de referência,
como os catálogos de identificadores BIC e FIRDS, são excluídas para que
cada acerto carregue proveniência genuína de sanções). Os apelidos foram
descartados para manter o arquivo abaixo de 2 MB.

Como a lista da OpenSanctions é uma ordem de magnitude maior que
nossa amostra anterior da OFAC, extraímos do Neo4j todos os nomes de dirigentes e entidades
*uma vez* e fazemos o join localmente contra o CSV de sanções usando
`PROC SQL`. A correspondência exata com UPCASE é robusta e evita os limites
de comprimento de string do Cypher que afligem grandes IN-lists de tokens.

In [ ]:
/* Lê o CSV incluído da OpenSanctions. Nove linhas de comentário de cabeçalho
   mais um cabeçalho de coluna = firstobs=11. */
dados opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    comprimento os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    entrada os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    se comprimento(os_name_upper) >= 6;
executar;

procedimento ordenar dados=opensanctions nodupkey out=os_dedup;
    por os_name_upper;
executar;

procedimento médias dados=os_dedup n;
    título "OpenSanctions sanctions-list records loaded";
executar;

/* Extrai todos os nomes de dirigentes + entidades do grafo. */
procedimento gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

procedimento gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Correspondência exata com UPCASE — dirigente para parte sancionada. */
procedimento sql;
    criar tabela s14_officer_hits as
    selecionar o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    de_tabela all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

procedimento sql;
    criar tabela s14_entity_hits as
    selecionar e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    de_tabela all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

procedimento sql;
    selecionar count(*) as n_officer_hits
    de_tabela s14_officer_hits;
    selecionar count(*) as n_entity_hits
    de_tabela s14_entity_hits;
quit;

procedimento imprimir dados=s14_officer_hits;
    título "Section 14 — officers on a consolidated sanctions list";
executar;

procedimento imprimir dados=s14_entity_hits;
    título "Section 14 — entities on a consolidated sanctions list";
executar;

## 15. Ranking composto de risco por entidade

Por fim, combinamos os sinais estruturais, jurisdicionais, relacionais e
de sanções calculados nas seções anteriores em um único
**score de risco** composto por entidade:

| Componente | Peso | Fonte |
|---|---:|---|
| Contagem de dirigentes (limitada a 50) | 0,25 | Tabela de atributos da Seção 5 |
| log(1 + PageRank do dirigente principal) | 0,25 | PageRank da Seção 4 + Seção 11 |
| Peso de risco da jurisdição | 0,25 | Tax Justice Network CTHI 2021 (incluída) |
| Sinalizador de dirigente sancionado | 0,25 | Resultados de correspondência exata da Seção 14 |

O risco de jurisdição vem do arquivo incluído
`data/tax_haven_ranks.csv`, montado a partir da lista de rankings publicados do
Corporate Tax Haven Index 2021 da Tax Justice Network. Os rankings 1-10 são
retirados diretamente do comunicado à imprensa do CTHI 2021; os rankings intermediários
são os valores publicados da metodologia da TJN para as demais
jurisdições que aparecem nos Paradise Papers. Jurisdições sem
ranking no CTHI (por exemplo, o marcador `XX`) recebem peso ≈ 0.

O relatório abaixo usa o `PROC REPORT` estilizado para um regulador. As
entidades no topo da lista são aquelas que, simultaneamente, (a)
estão domiciliadas em uma jurisdição de paraíso de primeira página, (b) têm muitos
dirigentes, (c) têm um dirigente com PageRank elevado E (d) têm ao menos
um dirigente sinalizado em uma lista consolidada de sanções.

In [ ]:
/* Carrega os rankings incluídos do Corporate Tax Haven Index 2021 da TJN.
   Oito linhas de comentário + duas linhas // a mais + um cabeçalho = firstobs=16. */
dados tax_haven;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    comprimento iso2 $5 jurisdiction_name $50;
    entrada iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
executar;

procedimento imprimir dados=tax_haven (obs=10);
    título "Section 15 — jurisdiction risk weights (CTHI 2021)";
executar;

/* Atributos por entidade com nome do dirigente principal e ano de constituição. */
procedimento gql conn=icij out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Tudo o que precisa se juntar (pagerank, peso de risco,
   sinalizador de dirigente sancionado) é feito em um único LEFT JOIN triplo
   com PROC SQL — sem cadeia de DATA MERGE BY, sem PROC SORTs redundantes,
   e o COALESCE nos dá os valores de fallback inline. */
procedimento gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

procedimento sql;
    criar tabela entity_flagged as
    selecionar e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case quando f.node_id is não null então 1 senão 0 fim
               as sanctioned_officer
    de_tabela   entity_full        as e
    left join officer_scored   as p
      on   e.top_officer_name = p.officer_name
    left join tax_haven       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* Risco composto: normaliza por min-max os atributos contínuos,
   ponderando-os em conjunto. PROC MEANS + um único DATA step é adequado
   para a normalização — isso é SAS idiomático. */
procedimento médias dados=entity_flagged noprint;
    variável top_officer_pr;
    saída out=pr_max_ds max=pr_max;
executar;

dados entity_score;
    se _n_ = 1 então definir pr_max_ds;
    definir entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    manter node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
executar;

/* Distribuição de risco em toda a população de 24.957 entidades. */
procedimento médias dados=entity_score n min mean max;
    variável risk officer_count risk_weight;
    título "Section 15 — risk distribution across all entities";
executar;

/* As 25 entidades de maior risco composto. O OUTOBS= do PROC SQL substitui um
   par PROC SORT + DATA step com obs=. */
procedimento sql outobs=25;
    título "Section 15 — top-25 composite-risk entities (names)";
    selecionar entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    de_tabela   entity_score
    order por risk desc;
quit;

/* Separadamente, revela as entidades ligadas a dirigentes sancionados. */
procedimento sql;
    título "Section 15 — entities with at least one sanctioned officer";
    selecionar entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    de_tabela   entity_score
    onde  sanctioned_officer = 1
    order por risk desc;
quit;

## 16. Classificação de jurisdições: canal vs. sumidouro

**Referência:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo et al. particionam o grafo global de propriedade corporativa
em "sink-OFCs" — onde o valor corporativo termina: BM, KY,
BVI, JE, IM — e "conduit-OFCs" — através dos quais o valor flui: NL,
UK, CH, SG, IE. O grafo dos Paradise Papers tem uma
população diferente (na maioria entidades domiciliadas na Appleby), mas a mesma
distinção estrutural deveria separar as jurisdições onde os
dirigentes se agrupam e os endereços terminam daquelas que principalmente
canalizam capital.

Para cada jurisdição, calculamos cinco atributos estruturais diretamente
a partir do grafo ativo:

| Atributo | Sinal de |
|---|---|
| `log(n_entity)` | tamanho absoluto da presença offshore da jurisdição |
| `avg_officers` | densidade de dirigentes por entidade (jurisdições sumidouro carregam mais dirigentes por entidade) |
| `avg_xborder_off` | contagem média de dirigentes cujo próprio país difere da jurisdição da entidade (indicador de canal) |
| `intermediary_share` | proporção de entidades com um vínculo de intermediário CONNECTED_TO |
| `address_share` | proporção de entidades com um vínculo REGISTERED_ADDRESS (indicador de sumidouro) |

Padronizamos para z-scores e, em seguida, aplicamos a clusterização
hierárquica de variância mínima de Ward. O `PROC TREE` renderiza o dendrograma. Note
que os nós Intermediary dos Paradise Papers se conectam aos nós Entity
via `CONNECTED_TO` — não `INTERMEDIARY_OF`, que existe no
esquema mas não carrega dados para este vazamento — por isso usamos
`CONNECTED_TO` aqui.

In [ ]:
/* Extrai os atributos estruturais por jurisdição do grafo ativo. */
procedimento gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Mantém apenas jurisdições com pelo menos dez entidades para que os
   z-scores padronizados sejam significativos.  O vazamento dos Paradise Papers
   tem 44 jurisdições no total; 23 atendem a esse limite. */
dados s16_filtered;
    definir s16_raw;
    se n_entity >= 10;
    log_n_entity = log(n_entity);
executar;

procedimento médias dados=s16_filtered noprint;
    variável log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    saída out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
executar;

dados s16_std;
    se _n_ = 1 então definir s16_stats;
    definir s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    manter jurisdiction z1 z2 z3 z4 z5;
executar;

procedimento imprimir dados=s16_std;
    título "Section 16 — standardised jurisdiction features";
executar;

/* Clusterização hierárquica de variância mínima de Ward. */
procedimento cluster dados=s16_std method=ward outtree=s16_tree;
    variável z1 z2 z3 z4 z5;
    id jurisdiction;
    título "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
executar;

/* Renderiza o dendrograma. */
procedimento tree dados=s16_tree horizontal;
    id jurisdiction;
    título "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
executar;

## 17. Componentes principais dos papéis dos dirigentes na rede

**Referência:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Ver também Newman M E J, *Networks: An Introduction* (Oxford, 2010),
capítulo 7.

Os dirigentes no grafo dos Paradise Papers desempenham papéis estruturalmente
diferentes. Alguns ocupam o centro de um grande cluster de entidades
relacionadas; outros ligam clusters de outro modo separados (corretores,
no sentido de Burt/Borgatti); alguns poucos formam o núcleo denso de uma
rede de insiders de uma jurisdição específica. Quatro métricas de grafo
capturam esses papéis distintos:

| Métrica | Captura |
|---|---|
| `degree` | Contagem de arestas de saída `OFFICER_OF` — amplitude de afiliação |
| `betweenness` | Fração de caminhos mais curtos que passam pelo dirigente — **corretagem** |
| `kcore` | Maior k para o qual o dirigente está em um subgrafo k-conectado — **imersão no núcleo** |
| `pagerank` | Score do tipo autovetor da mesma projeção — **influência via parceiros influentes** |

Calculamos todas as quatro métricas sobre a projeção não direcionada completa
`(Officer)—[OFFICER_OF]—(Entity)` por meio da
biblioteca Graph Data Science do Neo4j, restringimos aos 5.000 dirigentes
de maior grau e executamos o `PROC PRINCOMP` sobre as métricas com transformação logarítmica.

A PCA comprime as quatro métricas correlacionadas em eixos
ortogonais cujas cargas relativas nos dizem quais papéis se agrupam
empiricamente e quais são estruturalmente distintos.

*Nota sobre o coeficiente de clusterização local:* Garcia-Bernardo et al.
incluem o coeficiente de clusterização local como uma quinta métrica. A
projeção `(Officer)—[:OFFICER_OF]—(Entity)` dos Paradise Papers é
estritamente bipartida, portanto nenhum triângulo pode existir — todo coeficiente
de clusterização local é 0. Nós o descartamos do conjunto de métricas.

In [ ]:
/* O PROC NETWORK extrai um sub-grafo dos 5000 dirigentes principais por meio de um
   MATCH somente leitura e calcula grau, centralidade de autovetor e k-core
   em processo. Substitui um anterior gds.graph.project + quatro
   chamadas gds.<algorithm>.stream — esse padrão requer um passo de projeção
   GDS em modo de escrita que o Service step-neo4j somente leitura da
   plataforma rejeita.

   A centralidade de intermediação é intencionalmente omitida: o NetworkX
   calcula a intermediação exata em O(V·E), o que domina o tempo de execução
   neste sub-grafo (5.000 dirigentes × ~78.000 arestas). A PCA
   ainda tem três eixos ortogonais — grau (proeminência local),
   centralidade de autovetor (influência global) e k-core
   (coesão estrutural) — suficiente para separar os papéis dos dirigentes para
   a interpretação de destaque. */
procedimento network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar de_tabela=a_node_id até=b_node_id;
    centrality degree eigen=unweight;
    core;
executar;

/* Extrai ids/nomes dos nós de dirigentes para filtragem. */
procedimento gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Filtra para as linhas de dirigentes. A centralidade de autovetor substitui o
   PageRank — veja o comentário da Seção 4. */
procedimento sql;
    criar tabela s17_metrics as
    selecionar n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    de_tabela s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order por n.centr_degree desc;
quit;

dados s17_metrics;
    definir s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    manter node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
executar;

procedimento imprimir dados=s17_metrics (obs=10);
    título "Section 17 — top-10 officers by degree (raw + log metrics)";
executar;

procedimento médias dados=s17_metrics n mean std min max;
    variável log_degree log_pr k_val;
    título "Section 17 — log-transformed metric summary";
executar;

procedimento princomp dados=s17_metrics out=s17_scores;
    variável log_degree log_pr k_val;
    título "Section 17 — PCA of officer network roles";
executar;

procedimento sgplot dados=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis rótulo="PC1 (global influence axis)";
    yaxis rótulo="PC2 (brokerage vs core embeddedness)";
    título "Section 17 — officers in 2D principal-component role space";
executar;

## 18. Análise de intervenção ARIMA nas taxas de constituição

**Referência:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Aplicada aos vazamentos offshore por Koeppl T, Sipiczki I, Zhao H, *FinCEN
Files: Regulatory response and compliance* (2021).

A contagem anual de novas entidades no grafo dos Paradise Papers é uma
série de crescimento quase monotônico de 1970 (36 entidades) até 2007
(1.595 entidades, o pico), seguida de uma queda em 2008-2009 e um
platô mais lento até 2014 (o fim da cobertura do vazamento).

Aplicamos a clássica análise de intervenção de Box-Tiao para testar se
dois eventos do mundo real deixaram assinaturas mensuráveis na série de
constituições:

- **Degrau de 2009** — a repressão aos paraísos fiscais na cúpula do G20 em Londres
  (abril de 2009), que coincidiu com a crise financeira.
- **Degrau de 2014** — a FATCA (Foreign Account Tax Compliance Act) dos EUA
  entrou em vigor em 1º de julho de 2014.

A resposta é `log(n)`, portanto um coeficiente de intervenção de -0,30
corresponde a aproximadamente uma queda de 26% na taxa anual de constituições.
Ajustamos um `ARIMA(1,0,0)` — o modelo autorregressivo AR(1) sobre a
série fortemente correlacionada — com os dois indicadores de degrau como
variáveis exógenas `INPUT=`.

A hipótese nula é que nenhum dos degraus produz uma mudança
mensurável uma vez considerada a tendência AR(1). As metodologias
publicadas divergem sobre como interpretar uma não rejeição: pode
significar que a intervenção não teve efeito, ou que a
autocorrelação AR(1) está absorvendo o sinal da intervenção.

In [ ]:
procedimento gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Constrói o dataset da série de intervenção.  Dois indicadores de degrau:
   step_2009 = I{year >= 2009} captura a mudança de regime pré-anúncio
   do G20 Londres/FATCA; step_2014 = I{year >= 2014} captura
   o choque da data de vigência da FATCA.  A resposta é log(n), então
   as estimativas de coeficiente são lidas como efeitos proporcionais. */
dados s18_ts;
    definir year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
executar;

procedimento imprimir dados=s18_ts;
    título "Section 18 — annual new-entity formations + intervention dummies";
executar;

procedimento sgplot dados=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x rótulo="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x rótulo="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis rótulo="Incorporation year";
    yaxis rótulo="New entities per year";
    título "Section 18 — Paradise-Papers annual formations, 1970-2014";
executar;

/* Identifica o modelo e, em seguida, estima um ARIMA(1,0,0) com os dois
   inputs de degrau.  O CROSSCORR= do PROC ARIMA registra as variáveis exógenas
   no passo IDENTIFY para que fiquem disponíveis para o ESTIMATE INPUT=. */
procedimento arima dados=s18_ts;
    identify variável=log_n crosscorr=(step_2009 step_2014) nlag=8;
    estimativa p=1 entrada=(step_2009 step_2014);
    título "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
executar;
quit;

## 19. Modelo de contagem de exposição a sanções com inflação de zeros

**Referência:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, 2ª edição, Cambridge University Press (2013), capítulo 4.
Modelos com inflação de zeros: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

A Seção 14 encontrou apenas **cinco** entidades dos Paradise Papers com pelo
menos um dirigente em uma lista consolidada de sanções — um evento raríssimo.
Uma regressão Poisson ou binomial negativa padrão sobre
`sanctioned_count` por entidade se ajustaria mal porque **99,98%** das
entidades têm zero. O modelo binomial negativo com inflação de zeros (ZINB)
lida com isso dividindo o ajuste em duas partes:

1. Um modelo logístico para saber se a entidade pertence a uma classe de "zero
   estrutural" (nenhuma exposição possível a sanções).
2. Um modelo binomial negativo para a contagem entre as demais.

Com apenas 5 eventos positivos em 21.538 entidades, a verossimilhança
ZINB é degenerada — ambas as partes colapsam. Essa falha é uma
**propriedade honesta dos dados**, não do procedimento. Executamos o
ajuste ZINB mesmo assim para mostrar que o ferramental de regressão funciona de ponta a ponta,
e então recorremos a uma regressão logística binária comum sobre
`has_sanctioned` (indicador de `sanctioned_count > 0`). A
logística identifica um efeito claro: **cada unidade logarítmica adicional da
contagem de dirigentes multiplica as chances de ter pelo menos um
dirigente sancionado por cerca de 3,1** (p = 0,002).

Covariáveis:

- `top5` — variável de classe de 6 níveis (BM, KY, VG, IM, JE, OTHER),
  categoria de referência OTHER.
- `log_n_off` — `log(officer_count)`, o preditor de tamanho dominante.

In [ ]:
/* Extrai a contagem de dirigentes sancionados por entidade do grafo ativo. */
procedimento gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

dados s19;
    definir s19_raw;
    se officer_count >= 1;
    comprimento top5 $5;
    top5 = 'OTHER';
    se jurisdiction = 'BM' então top5 = 'BM';
    senão se jurisdiction = 'KY' então top5 = 'KY';
    senão se jurisdiction = 'VG' então top5 = 'VG';
    senão se jurisdiction = 'IM' então top5 = 'IM';
    senão se jurisdiction = 'JE' então top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
executar;

procedimento frequências dados=s19;
    tables top5 sanctioned_count has_sanctioned;
    título "Section 19 — response-variable distribution (very zero-heavy)";
executar;

/* ZINB primeiro — esperado ser degenerado em uma série de 5 eventos. */
procedimento genmod dados=s19;
    classe top5;
    modelo sanctioned_count = top5 log_n_off / dist=zinb link=log;
    título "Section 19 — ZINB count model (degenerate on 5 events)";
executar;

/* Fallback: logística binária sobre has_sanctioned.  Interpretável. */
procedimento logistic dados=s19 decrescente plots=none;
    classe top5;
    modelo has_sanctioned = top5 log_n_off;
    título "Section 19 — logistic regression (has-sanctioned fallback)";
executar;

## 20. Modelo Poisson de efeitos mistos para a taxa de constituição

**Referência:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Contagem em painel clássica: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

A Seção 18 ajustou um ARIMA univariado à série agregada de constituições.
Aqui usamos a dimensão de **painel**: uma linha por célula de
jurisdição-ano, ajustando um modelo linear generalizado misto (GLMM) de Poisson com
uma tendência linear fixa de ano mais um indicador de degrau da FATCA, e um **intercepto
aleatório por jurisdição**. Isso separa o efeito de tendência comum
do nível de cada jurisdição individual.

Restrição do painel: as 10 jurisdições cuja **contagem média anual**
é >=5 no período 1990-2014 (203 células no total). Jurisdições
menores com muitos anos de contagem zero desestabilizariam um
ajuste de Poisson.

O `PROC GLIMMIX` com `DIST=POISSON LINK=LOG` e
`RANDOM INTERCEPT / SUBJECT=jurisdiction` produz tanto os efeitos fixos
no nível populacional (tendência de ano, deslocamento FATCA) quanto o
componente de variância entre jurisdições. A variância do intercepto aleatório
nos diz *quanto as taxas de constituição diferem entre
jurisdições após considerar a tendência temporal comum* — um
score de heterogeneidade estrutural para a população de centros
financeiros offshore.

In [ ]:
/* Dataset de painel: células jurisdição x ano de 1990-2014. */
procedimento gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Mantém as jurisdições com contagem média anual >= 5. */
procedimento sql;
    criar tabela s20_keep_jur as
    selecionar jurisdiction, avg(n) as avg_n
    de_tabela s20_raw
    group por jurisdiction
    having avg(n) >= 5;
quit;

procedimento sql;
    criar tabela s20 as
    selecionar r.*,
           r.year - 2002 as year_c,
           case quando r.year >= 2014 então 1 senão 0 fim as fatca
    de_tabela s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

procedimento frequências dados=s20;
    tables jurisdiction;
    título "Section 20 — jurisdictions retained in the panel";
executar;

/* GLMM Poisson de efeitos mistos: tendência de ano fixa + degrau FATCA,
   intercepto aleatório por jurisdição. */
procedimento glimmix dados=s20;
    classe jurisdiction;
    modelo n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    título "Section 20 — mixed Poisson formation-rate model";
executar;

/* Efeitos de intercepto aleatório ordenados revelariam as
   jurisdições que sistematicamente superam ou ficam abaixo
   da tendência comum.  A instrução SOLUTION do PROC GLIMMIX imprime esses
   valores na saída padrão acima — deixamos o ranking para o
   leitor. */

In [ ]:
/* Fecha o PDF do relatório e libera a biblioteca do Neo4j. */
ods pdf close;

biblioteca icij clear;

## Reprodutibilidade e proveniência

- **Fonte dos dados do grafo:** base de dados de pesquisa *Offshore Leaks*
  da ICIJ, dataset *Paradise Papers*, lançado pela primeira vez em novembro de 2017.
  Disponível em <https://offshoreleaks.icij.org/pages/database>.
  Carregado no serviço compartilhado `step-neo4j` da plataforma
  (Neo4j 5.26 Community Edition, somente leitura no nível do servidor) por meio do
  dump público upstream em
  `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **Dados OFAC SDN:** exportação pública em CSV dos Specially Designated
  Nationals da OFAC do Tesouro dos EUA, obtida da API pública de
  preview do Tesouro em abril de 2026. O arquivo `data/ofac_sdn.csv` contém
  500 linhas curadas entre os cinco maiores programas da OFAC por contagem
  de entradas. Usado na triagem de dois estágios da Seção 6.
- **Dados OpenSanctions:** snapshot do dataset consolidado *default* da
  OpenSanctions de 17/04/2026, baixado de
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  O arquivo incluído `data/opensanctions_default.csv` contém as
  18.654 linhas com esquema de Pessoa e Empresa cujo dataset primário é
  uma das listas consolidadas de sanções da OFAC SDN, do HM Treasury do Reino Unido, das
  sanções financeiras da UE, do Conselho de Segurança da ONU, do Canadá, da Bélgica, da Austrália, da Suíça ou de outras
  listas nacionais consolidadas. Os apelidos foram descartados para
  manter o arquivo abaixo de 2 MB. Licença: ODbL 1.0 (OpenSanctions). Usado
  no enriquecimento da Seção 14.
- **Rankings de paraísos fiscais:** rankings publicados do *Corporate Tax Haven
  Index 2021* da Tax Justice Network, compilados da
  página inicial do índice `https://cthi.taxjustice.net` e do comunicado à
  imprensa de março de 2021 em
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  O arquivo incluído `data/tax_haven_ranks.csv` lista as
  jurisdições que aparecem nos Paradise Papers com seu
  ranking CTHI e um peso de risco derivado em `[0, 1]`. Licença: CC BY-SA
  4.0 (Tax Justice Network). Usado no ranking composto da Seção 15.
- **Algoritmos de grafo:** detecção de comunidades Louvain e
  centralidade de autovetor (o análogo interno mais próximo do PageRank)
  calculadas pelo `PROC NETWORK` em processo sobre listas de arestas extraídas via
  Cypher somente leitura. O Service compartilhado `step-neo4j` da plataforma é
  somente leitura no nível do servidor, portanto todos os algoritmos de grafo são executados no
  pod do workspace em vez de operações de escrita do Neo4j Graph Data
  Science.
- **Métodos estatísticos:** o `PROC LIFETEST` usa o estimador Kaplan-Meier
  com os testes estratificados log-rank, Wilcoxon e Tarone-Ware. O `PROC PHREG`
  ajusta o modelo de riscos proporcionais de Cox via
  empates de Breslow usando o wrapper Python/statsmodels. O `PROC LOGISTIC`
  ajusta uma regressão logística binária de máxima verossimilhança.
- **Métodos de composição de risco:** o score de influência composto da Seção 11
  normaliza grau, log-PageRank, amplitude jurisdicional
  e anos de atuação para `[0, 1]` e soma com pesos fixos
  (30/30/20/20). O score composto de risco por entidade da Seção 15
  normaliza a contagem de dirigentes limitada, o log-PageRank, o peso de risco do CTHI
  e um sinalizador binário de dirigente sancionado, somando com pesos iguais
  de 0,25 cada.

A análise completa é reproduzível a partir do script de smoke-test
nesta pasta: `./smoke.jenner`. Executá-lo de ponta a ponta contra o
serviço compartilhado `step-neo4j` (com `JENNER_NEO4J_HOST` e
`JENNER_NEO4J_PASS` definidos, o que a plataforma faz por você em qualquer
pod do workspace) leva cerca de dois minutos e verifica que cada
consulta e cada PROC — incluindo as cinco novas seções adicionadas
junto ao pipeline existente — retornam a saída esperada no
grafo real de 163.435 nós.